In [2]:
import requests
import os
import sys
import platform
from lakehouse.spark import bronze, silver, gold
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [3]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [4]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [5]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [6]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

DataFrame[]

In [7]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [8]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [9]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [10]:
bronze_instance.load().transform().write(mode="overwrite").execute("people")

2025-03-16 07:35:10 | people | execute | Started
2025-03-16 07:35:10 | people | execute | Started
2025-03-16 07:35:10 | people | load | Started
2025-03-16 07:35:16 | people | load | Completed in 0.08 min
2025-03-16 07:35:16 | people | transform | Started
2025-03-16 07:35:16 | people | transform | Completed in 0.0 min
2025-03-16 07:35:16 | people | write | Started
2025-03-16 07:35:38 | people | write | Completed in 0.37 min
2025-03-16 07:35:38 | people | execute | Completed in 0.45 min
2025-03-16 07:35:38 | people | execute | Completed in 0.45 min


In [11]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+-------------------+---+--------------------+--------------------+
|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+-------------------+---+--------------------+--------------------+
|2025-03-16 07:35:...|        Cliegg Lars| 62|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:35:...|  Poggle the Lesser| 63|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:35:...|    Luminara Unduli| 64|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:35:...|      Barriss Offee| 65|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:35:...|              Dormé| 66|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:35:...|              Dooku| 67|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:35:...|Bail Prestor Organa| 68|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:35:...|         Jango Fett| 69|https://www.swapi...|{"created": "2025...|
|2025-03

# 2 Silver

In [12]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [13]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

In [14]:
class StarWarsSilver(silver.Silver):
    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        if table == "people":
            df = self.transf_people(df)
        return df

    def transf_people(self, df: DataFrame) -> DataFrame:
        df = (
            df.withColumn("height", df.properties.height)
            .withColumn("mass", df.properties.mass)
            .withColumn("gender", df.properties.gender)
            .drop("url", "properties")
        )
        return df


silver_instance = StarWarsSilver(spark, **options)

In [15]:
silver_instance.load().transform().write(mode="overwrite", merge_schema=True).execute(
    "people"
)

2025-03-16 07:35:40 | people | execute | Started
2025-03-16 07:35:40 | people | execute | Started
2025-03-16 07:35:40 | people | load | Started
2025-03-16 07:35:40 | people | load | Completed in 0.0 min
2025-03-16 07:35:40 | people | transform | Started
2025-03-16 07:35:40 | people | transform | Completed in 0.0 min
2025-03-16 07:35:40 | people | write | Started
2025-03-16 07:35:42 | people | write | Completed in 0.02 min
2025-03-16 07:35:42 | people | execute | Completed in 0.02 min
2025-03-16 07:35:42 | people | execute | Completed in 0.02 min


In [16]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 82
+--------------------------+-------------------------+---------------------+---+-------+-------+-------------+
|LH_SilverTS               |LH_BronzeTS              |name                 |uid|height |mass   |gender       |
+--------------------------+-------------------------+---------------------+---+-------+-------+-------------+
|2025-03-16 07:35:40.965646|2025-03-16 07:35:17.38251|Cliegg Lars          |62 |183    |unknown|male         |
|2025-03-16 07:35:40.965646|2025-03-16 07:35:17.38251|Poggle the Lesser    |63 |183    |80     |male         |
|2025-03-16 07:35:40.965646|2025-03-16 07:35:17.38251|Luminara Unduli      |64 |170    |56.2   |female       |
|2025-03-16 07:35:40.965646|2025-03-16 07:35:17.38251|Barriss Offee        |65 |166    |50     |female       |
|2025-03-16 07:35:40.965646|2025-03-16 07:35:17.38251|Dormé                |66 |165    |unknown|female       |
|2025-03-16 07:35:40.965646|2025-03-16 07:35:17.38251|Dooku                |67 |193    |80     |mal

# 3 Gold

In [17]:
options = {
    "catalog": CATALOG,
    "source_schema": "silver",
    "target_schema": "gold",
}

In [18]:
class StarWarsGold(gold.Gold):
    def people_per_gender(self, df: DataFrame, table: str) -> DataFrame:
        df = df.where("gender <> 'n/a'")
        df = df.where("gender <> 'none'")
        df = df.groupBy("gender").count()
        return df

    def all_females(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("gender = 'female'").drop("LH_SilverTS", "LH_BronzeTS")


gold_instance = StarWarsGold(spark, **options)

In [19]:
gold_instance.load(source_tbl="people").transform(
    tbl_transformations={
        "peoplegender": "people_per_gender",
        "peoplefemale": "all_females",
    }
).write(mode="overwrite", merge_schema=True).execute("peoplegender", "peoplefemale")

2025-03-16 07:35:44 | peoplegender | execute | Started
2025-03-16 07:35:44 | peoplegender | execute | Started
2025-03-16 07:35:44 | peoplegender | load | Started
2025-03-16 07:35:44 | peoplegender | load | Completed in 0.0 min
2025-03-16 07:35:44 | peoplegender | transform | Started
2025-03-16 07:35:44 | peoplegender | transform | Completed in 0.0 min
2025-03-16 07:35:44 | peoplegender | write | Started
2025-03-16 07:35:46 | peoplegender | write | Completed in 0.03 min
2025-03-16 07:35:46 | peoplegender | execute | Completed in 0.03 min
2025-03-16 07:35:46 | peoplefemale | execute | Started
2025-03-16 07:35:46 | peoplefemale | load | Started
2025-03-16 07:35:46 | peoplefemale | load | Completed in 0.0 min
2025-03-16 07:35:46 | peoplefemale | transform | Started
2025-03-16 07:35:46 | peoplefemale | transform | Completed in 0.0 min
2025-03-16 07:35:46 | peoplefemale | write | Started
2025-03-16 07:35:48 | peoplefemale | write | Completed in 0.02 min
2025-03-16 07:35:48 | peoplefemale | e

In [20]:
df = spark.sql(f"SELECT * FROM {CATALOG}.gold.peoplegender")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 3
+--------------------------+-------------+-----+
|LH_GoldTS                 |gender       |count|
+--------------------------+-------------+-----+
|2025-03-16 07:35:44.128752|female       |17   |
|2025-03-16 07:35:44.128752|male         |60   |
|2025-03-16 07:35:44.128752|hermaphrodite|1    |
+--------------------------+-------------+-----+



In [21]:
df = spark.sql(f"SELECT * FROM {CATALOG}.gold.peoplefemale")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 17
+--------------------------+------------------+---+------+-------+------+
|LH_GoldTS                 |name              |uid|height|mass   |gender|
+--------------------------+------------------+---+------+-------+------+
|2025-03-16 07:35:46.383093|Luminara Unduli   |64 |170   |56.2   |female|
|2025-03-16 07:35:46.383093|Barriss Offee     |65 |166   |50     |female|
|2025-03-16 07:35:46.383093|Dormé             |66 |165   |unknown|female|
|2025-03-16 07:35:46.383093|Zam Wesell        |70 |168   |55     |female|
|2025-03-16 07:35:46.383093|Taun We           |73 |213   |unknown|female|
|2025-03-16 07:35:46.383093|Jocasta Nu        |74 |167   |unknown|female|
|2025-03-16 07:35:46.383093|R4-P17            |75 |96    |unknown|female|
|2025-03-16 07:35:46.383093|Shaak Ti          |78 |178   |57     |female|
|2025-03-16 07:35:46.383093|Sly Moore         |82 |178   |48     |female|
|2025-03-16 07:35:46.383093|Shmi Skywalker    |43 |163   |unknown|female|
|2025-03-16 07:35:46.3830

# 4 Clean Up

In [ ]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.gold CASCADE")
spark.stop()

DataFrame[]